# 01 - What is SAPIENT?

## Learning Objectives

By the end of this notebook, you will:

1. Understand the history and motivation behind SAPIENT
2. Know the key message types in the SAPIENT protocol
3. Understand the SAPIENT architecture (nodes, sensors, fusion)
4. Recognize the benefits of standardization in sensor networks
5. Understand how SAPIENT compares to proprietary protocols

---

## The Problem: Pre-SAPIENT World

### Scenario: A Security Operation

Imagine you're protecting a critical infrastructure site:

```
┌─────────────────────────────────────────────────────┐
│                 Protected Facility                  │
│                                                     │
│  Sensors Deployed:                                  │
│  • 3 × Ground Surveillance Radars (Vendor A)        │
│  • 4 × Thermal Cameras (Vendor B)                   │
│  • 6 × Acoustic Sensors (Vendor C)                  │
│  • 2 × RF Detectors (Vendor D)                      │
└─────────────────────────────────────────────────────┘
```

### The Integration Nightmare

Each vendor provides:
- **Different message formats** (XML, JSON, binary proprietary)
- **Different protocols** (TCP, UDP, HTTP, custom)
- **Different coordinate systems** (UTM, lat/lon, local grid)
- **Different data models** (tracks, detections, alerts)

#### Cost of Integration

Without a standard:
- 🕐 **Time**: 6-12 months per sensor type
- 💷 **Money**: £100k-500k per integration
- 🔧 **Maintenance**: Custom adapters for every sensor
- 🔒 **Lock-in**: Can't easily swap vendors
- ⚠️ **Risk**: Each adapter is a failure point

## The Solution: SAPIENT

### What SAPIENT Provides

**SAPIENT = Sensors And Processing for INTelligence**

A standardized protocol that defines:
1. **Message formats** - Structured using Protocol Buffers
2. **Message types** - Detection, Status, Task, etc.
3. **Communication patterns** - Request/response, pub/sub
4. **Coordinate systems** - Standardized reference frames
5. **Metadata standards** - Common fields across all sensors

### The SAPIENT Architecture

```
┌────────────────────────────────────────────────────────┐
│                    Command Layer                       │
│         (Operators, Decision Support Systems)          │
└────────────────────┬───────────────────────────────────┘
                     │ SAPIENT Messages
                     ▼
┌────────────────────────────────────────────────────────┐
│                   Fusion Layer                         │
│  • Combines data from multiple sensors                 │
│  • Correlates detections into tracks                   │
│  • Resolves conflicts and ambiguities                  │
└─────┬────────────────┬─────────────────┬───────────────┘
      │                │                 │
      │ SAPIENT        │ SAPIENT         │ SAPIENT
      ▼                ▼                 ▼
┌─────────┐      ┌─────────┐      ┌─────────┐
│ Radar   │      │ Camera  │      │ Acoustic│
│ Node    │      │ Node    │      │ Node    │
└─────────┘      └─────────┘      └─────────┘
```

### Key Insight

With SAPIENT, adding a new sensor is **plug-and-play**:
- Sensor speaks SAPIENT → Works immediately
- No custom adapter needed
- Vendor-agnostic
- Interoperable by design

## SAPIENT Message Types

SAPIENT defines several core message types. Let's explore each one.

### 1. Registration Message

**Purpose**: Sensor announces itself to the fusion node

**Content**:
- Sensor ID and type (radar, camera, etc.)
- Capabilities (detection types, coverage area)
- Location and orientation

**Example Flow**:
```
Sensor → Fusion: "I'm radar-001, I detect vehicles and people, 
                  covering 0-180° azimuth, 50-5000m range"
```

### 2. Status Message

**Purpose**: Sensor reports its health and operational state

**Content**:
- Operational status (active, degraded, failed)
- Battery/power level
- Self-test results
- Active mode (searching, tracking, idle)

**Example Flow**:
```
Sensor → Fusion: "Status: Active, Battery: 78%, Mode: Searching"
```

### 3. Detection Message

**Purpose**: Sensor reports something it detected

**Content**:
- Location (range, bearing, elevation)
- Classification (vehicle, person, unknown)
- Confidence level
- Sensor-specific data (RCS for radar, pixel coordinates for camera)

**Example Flow**:
```
Sensor → Fusion: "Detection at (1500m, 45°), 
                  Class: Vehicle, Confidence: 0.89"
```

### 4. Task Message

**Purpose**: Fusion node assigns a task to a sensor

**Content**:
- Task type (search area, track target, change mode)
- Parameters (coordinates, priorities)
- Duration or trigger conditions

**Example Flow**:
```
Fusion → Sensor: "Task: Track target at (2000m, 30°), 
                  Update every 1 second"
```

### 5. Task ACK Message

**Purpose**: Sensor acknowledges receipt of a task

**Content**:
- Task ID being acknowledged
- Acceptance status (accepted, rejected, cannot comply)
- Reason for rejection if applicable

**Example Flow**:
```
Sensor → Fusion: "Task 12345: Accepted, beginning tracking"
```

## Let's Model These Messages in Python

We'll create educational simplified versions to understand the structure.

**Note**: These are NOT the official SAPIENT message formats. Those use Protocol Buffers, which we'll introduce in notebook 03.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Optional
from uuid import uuid4


class SensorType(Enum):
    """Types of sensors in a SAPIENT network."""
    RADAR = "radar"
    CAMERA = "camera"
    ACOUSTIC = "acoustic"
    RF = "rf"
    LIDAR = "lidar"


class Status(Enum):
    """Operational status of a sensor."""
    ACTIVE = "active"
    DEGRADED = "degraded"
    FAILED = "failed"
    INITIALIZING = "initializing"


class ClassificationType(Enum):
    """Object classification types."""
    PERSON = "person"
    VEHICLE = "vehicle"
    AIRCRAFT = "aircraft"
    UNKNOWN = "unknown"


@dataclass
class Location:
    """Represents a detected object's location."""
    range_m: float  # Distance from sensor in meters
    azimuth_deg: float  # Bearing from sensor in degrees (0° = North)
    elevation_deg: float = 0.0  # Elevation angle in degrees


print("✅ Base types and enums defined")

### Registration Message Model

In [ ]:
@dataclass
class RegistrationMessage:
    """
    Simplified SAPIENT Registration message.
    
    A sensor sends this when it first connects to announce its
    capabilities and configuration.
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "unknown-sensor"
    sensor_type: SensorType = SensorType.RADAR
    
    # Coverage capabilities
    min_range_m: float = 0.0
    max_range_m: float = 5000.0
    azimuth_start_deg: float = 0.0
    azimuth_end_deg: float = 360.0
    
    def __str__(self) -> str:
        return (
            f"REGISTRATION: {self.sender_id} ({self.sensor_type.value})\n"
            f"  Range: {self.min_range_m:.0f}-{self.max_range_m:.0f}m\n"
            f"  Azimuth: {self.azimuth_start_deg:.0f}-{self.azimuth_end_deg:.0f}°"
        )


# Example: Radar registering itself
radar_reg = RegistrationMessage(
    sender_id="radar-001",
    sensor_type=SensorType.RADAR,
    min_range_m=50.0,
    max_range_m=5000.0,
    azimuth_start_deg=0.0,
    azimuth_end_deg=180.0
)

print(radar_reg)

### Status Message Model

In [ ]:
@dataclass
class StatusMessage:
    """
    Simplified SAPIENT Status message.
    
    Sensors periodically send these to report their health.
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "unknown-sensor"
    status: Status = Status.ACTIVE
    battery_percent: Optional[float] = None
    info: str = ""
    
    def __str__(self) -> str:
        battery_str = f", Battery: {self.battery_percent:.0f}%" if self.battery_percent else ""
        info_str = f" - {self.info}" if self.info else ""
        return (
            f"STATUS: {self.sender_id} = {self.status.value.upper()}"
            f"{battery_str}{info_str}"
        )


# Example: Radar reporting healthy status
status_msg = StatusMessage(
    sender_id="radar-001",
    status=Status.ACTIVE,
    battery_percent=85.0,
    info="Scanning mode"
)

print(status_msg)

### Detection Message Model

In [ ]:
@dataclass
class DetectionMessage:
    """
    Simplified SAPIENT Detection message.
    
    Sensors send these when they detect an object.
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "unknown-sensor"
    
    location: Location = field(default_factory=lambda: Location(0, 0))
    classification: ClassificationType = ClassificationType.UNKNOWN
    confidence: float = 0.0  # 0.0 to 1.0
    
    # Optional sensor-specific fields
    velocity_mps: Optional[float] = None  # Meters per second
    
    def __str__(self) -> str:
        vel_str = f", Velocity: {self.velocity_mps:.1f} m/s" if self.velocity_mps else ""
        return (
            f"DETECTION: {self.sender_id}\n"
            f"  Location: {self.location.range_m:.0f}m @ {self.location.azimuth_deg:.0f}°\n"
            f"  Class: {self.classification.value} (confidence: {self.confidence:.2f}){vel_str}"
        )


# Example: Radar detecting a vehicle
detection = DetectionMessage(
    sender_id="radar-001",
    location=Location(range_m=1500.0, azimuth_deg=45.0),
    classification=ClassificationType.VEHICLE,
    confidence=0.89,
    velocity_mps=12.5
)

print(detection)

### Task Message Model

In [ ]:
class TaskType(Enum):
    """Types of tasks that can be assigned to sensors."""
    SEARCH_AREA = "search_area"
    TRACK_TARGET = "track_target"
    CHANGE_MODE = "change_mode"


@dataclass
class TaskMessage:
    """
    Simplified SAPIENT Task message.
    
    Fusion nodes send these to direct sensor behavior.
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "fusion-node"
    recipient_id: str = "unknown-sensor"
    
    task_id: str = field(default_factory=lambda: str(uuid4()))
    task_type: TaskType = TaskType.SEARCH_AREA
    parameters: dict = field(default_factory=dict)
    
    def __str__(self) -> str:
        params_str = ", ".join(f"{k}={v}" for k, v in self.parameters.items())
        return (
            f"TASK: {self.sender_id} → {self.recipient_id}\n"
            f"  Task ID: {self.task_id}\n"
            f"  Type: {self.task_type.value}\n"
            f"  Parameters: {params_str}"
        )


# Example: Fusion node asking radar to track a specific target
task = TaskMessage(
    sender_id="fusion-001",
    recipient_id="radar-001",
    task_type=TaskType.TRACK_TARGET,
    parameters={
        "target_range_m": 1500.0,
        "target_azimuth_deg": 45.0,
        "update_rate_hz": 1.0
    }
)

print(task)

### Task Acknowledgement Message Model

In [ ]:
class TaskStatus(Enum):
    """Status codes for task acknowledgements."""
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    COMPLETED = "completed"
    FAILED = "failed"


@dataclass
class TaskAckMessage:
    """
    Simplified SAPIENT Task Acknowledgement message.
    
    Sensors send these in response to task assignments.
    """
    message_id: str = field(default_factory=lambda: str(uuid4()))
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    sender_id: str = "unknown-sensor"
    
    task_id: str = ""  # References the task being acknowledged
    task_status: TaskStatus = TaskStatus.ACCEPTED
    reason: str = ""
    
    def __str__(self) -> str:
        reason_str = f" - {self.reason}" if self.reason else ""
        return (
            f"TASK ACK: {self.sender_id}\n"
            f"  Task ID: {self.task_id}\n"
            f"  Status: {self.task_status.value.upper()}{reason_str}"
        )


# Example: Radar accepting the tracking task
task_ack = TaskAckMessage(
    sender_id="radar-001",
    task_id=task.task_id,  # Reference the task from above
    task_status=TaskStatus.ACCEPTED,
    reason="Beginning target tracking"
)

print(task_ack)

## A Complete Message Exchange

Let's simulate a realistic sequence of SAPIENT messages.

In [ ]:
import time

def print_message_exchange():
    """
    Simulate a realistic SAPIENT message exchange.
    """
    print("="*70)
    print("SAPIENT Message Exchange Simulation")
    print("="*70)
    print()
    
    # 1. Sensor comes online and registers
    print("[T+0s] Sensor connects and registers...")
    reg = RegistrationMessage(
        sender_id="radar-001",
        sensor_type=SensorType.RADAR,
        max_range_m=5000.0,
        azimuth_end_deg=180.0
    )
    print(reg)
    print()
    
    # 2. Sensor sends initial status
    print("[T+1s] Sensor reports initial status...")
    status = StatusMessage(
        sender_id="radar-001",
        status=Status.ACTIVE,
        battery_percent=100.0,
        info="Ready for tasking"
    )
    print(status)
    print()
    
    # 3. Sensor detects something
    print("[T+5s] Sensor makes a detection...")
    det = DetectionMessage(
        sender_id="radar-001",
        location=Location(1200.0, 35.0),
        classification=ClassificationType.VEHICLE,
        confidence=0.82,
        velocity_mps=15.0
    )
    print(det)
    print()
    
    # 4. Fusion node assigns tracking task
    print("[T+6s] Fusion node assigns tracking task...")
    task = TaskMessage(
        sender_id="fusion-001",
        recipient_id="radar-001",
        task_type=TaskType.TRACK_TARGET,
        parameters={
            "target_range_m": 1200.0,
            "target_azimuth_deg": 35.0,
            "update_rate_hz": 2.0
        }
    )
    print(task)
    print()
    
    # 5. Sensor acknowledges task
    print("[T+6.1s] Sensor acknowledges task...")
    ack = TaskAckMessage(
        sender_id="radar-001",
        task_id=task.task_id,
        task_status=TaskStatus.ACCEPTED,
        reason="Transitioning to tracking mode"
    )
    print(ack)
    print()
    
    # 6. Sensor provides tracking updates
    print("[T+7s] Sensor provides first tracking update...")
    track_update = DetectionMessage(
        sender_id="radar-001",
        location=Location(1180.0, 34.5),  # Target moved closer
        classification=ClassificationType.VEHICLE,
        confidence=0.91,  # Higher confidence in tracking mode
        velocity_mps=14.8
    )
    print(track_update)
    print()
    
    print("="*70)
    print("This is a typical SAPIENT interaction pattern!")
    print("="*70)

print_message_exchange()

## Key Benefits of SAPIENT

### 1. Interoperability
Any SAPIENT-compliant sensor works with any SAPIENT-compliant fusion node.

### 2. Vendor Independence
You're not locked into a single vendor's ecosystem.

### 3. Reduced Integration Time
Weeks instead of months to add a new sensor.

### 4. Lower Costs
No custom adapters = less development and maintenance.

### 5. Improved Reliability
Standardized protocol means fewer bugs and edge cases.

### 6. Simplified Testing
Test once against the standard, not against every sensor.

## Real-World Analogy

Think of SAPIENT like **USB**:

### Before USB (Pre-SAPIENT)
```
Computer has:
- PS/2 port (keyboard/mouse)
- Serial port (modem)
- Parallel port (printer)
- Game port (joystick)

Each device needs:
- Different cable
- Different driver
- Different protocol
```

### After USB (Post-SAPIENT)
```
Computer has:
- USB port

All devices:
- Same connector
- Standard protocol
- Plug and play
```

**SAPIENT is the "USB" for sensor networks.**

## Architecture Deep Dive

### Sensor Node (ASM - Advanced Sensor Manager)

The sensor node:
1. **Interfaces with the physical sensor** (radar hardware, camera, etc.)
2. **Translates sensor data** into SAPIENT messages
3. **Manages sensor state** (modes, tasking, health)
4. **Handles communication** with fusion nodes

```
┌─────────────────────────────────────┐
│        Sensor Node (ASM)            │
│  ┌─────────────────────────────┐    │
│  │  SAPIENT Protocol Handler   │    │
│  └───────────┬─────────────────┘    │
│              │                      │
│  ┌───────────▼─────────────────┐    │
│  │  Message Generator          │    │
│  └───────────┬─────────────────┘    │
│              │                      │
│  ┌───────────▼─────────────────┐    │
│  │  Sensor Hardware Interface  │    │
│  └───────────┬─────────────────┘    │
└──────────────┼──────────────────────┘
               │
          ┌────▼────┐
          │  Radar  │
          │ Hardware│
          └─────────┘
```

### Fusion Node (SDA - Sensor Data Application)

The fusion node:
1. **Receives detections** from multiple sensors
2. **Correlates detections** into coherent tracks
3. **Resolves conflicts** between sensors
4. **Tasks sensors** for optimal coverage
5. **Provides tracks** to command systems

```
┌─────────────────────────────────────┐
│       Fusion Node (SDA)             │
│  ┌─────────────────────────────┐    │
│  │  Command Interface          │    │
│  └───────────┬─────────────────┘    │
│              │                      │
│  ┌───────────▼─────────────────┐    │
│  │  Track Correlation          │    │
│  └───────────┬─────────────────┘    │
│              │                      │
│  ┌───────────▼─────────────────┐    │
│  │  Sensor Management          │    │
│  └───────────┬─────────────────┘    │
│              │                      │
│  ┌───────────▼─────────────────┐    │
│  │  SAPIENT Message Handler    │    │
│  └─────────────────────────────┘    │
└─────────────────────────────────────┘
```

## Summary

### What is SAPIENT?
A standardized protocol for sensor networks that enables:
- Plug-and-play sensor integration
- Vendor independence
- Reduced costs and complexity

### Key Message Types
1. **Registration** - Sensor announces itself
2. **Status** - Sensor reports health
3. **Detection** - Sensor reports what it sees
4. **Task** - Fusion directs sensor behavior
5. **Task ACK** - Sensor acknowledges tasks

### Architecture
- **Sensor Nodes (ASM)** translate sensor data to SAPIENT
- **Fusion Nodes (SDA)** combine data from multiple sensors
- **Command Systems** use fused data for decision-making

## Exercises

### Exercise 1: Create a Camera Detection

Model a camera detecting a person at 500m range, 120° azimuth, classified as "person" with 0.95 confidence.

In [ ]:
# Your solution here


### Exercise 2: Sensor Failure Scenario

Create a status message from "camera-003" reporting a DEGRADED status with 15% battery remaining and info="Lens condensation detected".

In [ ]:
# Your solution here


### Exercise 3: Task Rejection

Create a TaskAckMessage where a sensor rejects a task because the target is outside its coverage area.

In [ ]:
# Your solution here


### Exercise 4: Multi-Sensor Scenario

Consider this scenario:
- Radar detects a target at (2000m, 45°), classified as VEHICLE, confidence 0.70
- Camera at the same site detects at (2010m, 46°), classified as VEHICLE, confidence 0.92

Questions:
1. Are these the same target? How would you decide?
2. What would you report to the command system?
3. What is the benefit of having both sensors?

In [ ]:
# Create both detections

# Your analysis here


## What's Next?

In **02_System_Architecture.ipynb**, we'll:
- Design a complete SAPIENT system
- Explore network topology options
- Understand discovery and registration flows
- Plan our implementation architecture

---

**Previous:** [00_Introduction.ipynb](./00_Introduction.ipynb)  
**Next:** [02_System_Architecture.ipynb](./02_System_Architecture.ipynb)